<a href="https://colab.research.google.com/github/theMorana/HSE/blob/main/HSE/Machine%20Learning/Semester_2_Homeworks%20/ML_Hometask_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Задача

1. Собрать свой текстовый корпус (не менее 500 примеров) с помощью библиотеки `requests` или другого инструмента парсинга.
2. Выполнить разметку собранных данных для задачи классификации (бинарной или многоклассовой).
3. Обучить свёрточную нейросеть (CNN) на собранных и размеченных данных с использованием PyTorch или TensorFlow на выбор.
4. Самостоятельно подобрать архитектуру модели и гиперпараметры (количество слоёв, размер ядра, функцию активации, оптимизатор и т.д.).
5. После обучения вывести на экран метрики качества (accuracy, precision, recall, F1-score) на тестовой выборке.

Мини предыстория: я хотела изначально скрейпить посты с reddit по Baldur's Gate 3, но он меня постоянно блочил, и в целом я смогла выгрузить всего 25 постов, так что я решила сменить ресурс и парсила Steam )

##Вот код, где меня блочит редит

In [ ]:
!pip install requests beautifulsoup4

import requests
from bs4 import BeautifulSoup
headers = {"User-Agent": "python:reddit-scraper:v1.0 (by /u/yourname)"}
r = requests.get("https://old.reddit.com/r/BaldursGate3/", headers=headers)
soup = BeautifulSoup(r.text, "html.parser")

# user define function
# Scrape the data
def getdata(url):
    r = requests.get(url, headers = HEADERS)
    return r.text

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    ),
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Referer": "https://www.reddit.com/",
    "DNT": "1",
    "Connection": "keep-alive",
}

url = "https://old.reddit.com/r/BaldursGate3/"

# pass the url
# into getdata function
htmldata = getdata(url)
soup = BeautifulSoup(htmldata, 'html.parser')

# display html code
print(soup)

<!DOCTYPE html>

<html>
<head>
<title>Blocked</title>
<style>
      body {
          font: small verdana, arial, helvetica, sans-serif;
          width: 600px;
          margin: 0 auto;
      }

      h1 {
          height: 40px;
          background: transparent url(//www.redditstatic.com/reddit.com.header.png) no-repeat scroll top right;
      }
    </style>
</head>
<body>
<h1>whoa there, pardner!</h1>
<p>Your request has been blocked due to a network policy.</p>
<p>Try logging in or creating an account <a href="https://www.reddit.com/login/">here</a> to get back to browsing.</p>
<p>If you're running a script or application, please register or sign in with your developer credentials <a href="https://www.reddit.com/wiki/api/">here</a>. Additionally make sure your User-Agent is not empty and is something unique and descriptive and try again. if you're supplying an alternate User-Agent string,
try changing back to default as that can sometimes result in a block.</p>
<p>You can read Redd

## 1. Загрузка датасета

In [ ]:
!pip install requests

In [ ]:
import requests
import json
import csv
import time
from datetime import datetime
from google.colab import files

Вот здесь ИИ помог мне правильно спарсить сайт, подсказав нужную ссылку и метод (можете увидеть, что я используй общий сайт и код обсуждения игры, так как просто сайт с обсуждением парсился не так хорошо)

In [ ]:
APP_ID = 1086940
BASE_URL = "https://store.steampowered.com/appreviews/"

def fetch_reviews(cursor="*", num_per_page=100, language="english", review_type="all"):
    """
    review_type: 'all', 'positive', 'negative'
    """
    params = {
        "json": 1,
        "cursor": cursor,
        "num_per_page": num_per_page,
        "language": language,
        "filter": "recent",
        "purchase_type": "all",
        "review_type": review_type,   #параметр для балансировки
    }
    url = f"{BASE_URL}{APP_ID}"
    response = requests.get(url, params=params)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Ошибка {response.status_code}")
        return None

def parse_reviews(reviews_data):
    parsed = []
    for review in reviews_data.get("reviews", []):
        parsed.append({
            "recommend": review.get("voted_up", False),
            "review_text": review.get("review", "")
        })
    return parsed

def save_to_csv(reviews, filename="baldurs_gate_3_reviews_balanced.csv"):
    if not reviews:
        print("Нет отзывов для сохранения")
        return
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=reviews[0].keys())
        writer.writeheader()
        writer.writerows(reviews)
    print(f"Сохранено {len(reviews)} отзывов в {filename}")

def collect_balanced_reviews(target_count=500, min_text_len=20):
    """
    Собрать target_count положительных и target_count отрицательных отзывов.
    min_text_len - минимальная длина текста (отсекает слишком короткие)
    """
    positive = []
    negative = []

    # положительные отзывы
    print("Сбор положительных отзывов (рекомендую)...")
    cursor = "*"
    while len(positive) < target_count:
        data = fetch_reviews(cursor=cursor, review_type="positive")
        if not data or not data.get("reviews"):
            break
        new_reviews = parse_reviews(data)
        filtered = [r for r in new_reviews if len(r["review_text"].strip()) >= min_text_len]
        positive.extend(filtered)
        print(f"  Положительных: {len(positive)}/{target_count}")
        cursor = data.get("cursor")
        if not cursor:
            break
        time.sleep(0.5)

    # отрицательные отзывы
    print("Сбор отрицательных отзывов (не рекомендую)...")
    cursor = "*"
    while len(negative) < target_count:
        data = fetch_reviews(cursor=cursor, review_type="negative")
        if not data or not data.get("reviews"):
            break
        new_reviews = parse_reviews(data)
        filtered = [r for r in new_reviews if len(r["review_text"].strip()) >= min_text_len]
        negative.extend(filtered)
        print(f"  Отрицательных: {len(negative)}/{target_count}")
        cursor = data.get("cursor")
        if not cursor:
            break
        time.sleep(0.5)

    # Обрезаем до точного target_count (если собрали больше)
    positive = positive[:target_count]
    negative = negative[:target_count]

    # Объединяем и перемешиваем для случайного порядка
    balanced_reviews = positive + negative
    import random
    random.shuffle(balanced_reviews)

    print(f"\nИтог: {len(positive)} положительных, {len(negative)} отрицательных")
    return balanced_reviews

if __name__ == "__main__":
    # сбор по 500 отзывов каждого класса (всего 1000)
    reviews_balanced = collect_balanced_reviews(target_count=500, min_text_len=20)
    save_to_csv(reviews_balanced)

Сбор положительных отзывов (рекомендую)...
  Положительных: 69/500
  Положительных: 146/500
  Положительных: 216/500
  Положительных: 289/500
  Положительных: 362/500
  Положительных: 429/500
  Положительных: 501/500
Сбор отрицательных отзывов (не рекомендую)...
  Отрицательных: 90/500
  Отрицательных: 176/500
  Отрицательных: 269/500
  Отрицательных: 355/500
  Отрицательных: 443/500
  Отрицательных: 535/500

Итог: 500 положительных, 500 отрицательных
Сохранено 1000 отзывов в baldurs_gate_3_reviews_balanced.csv


Так как в Steam есть метки "рекомендую" и "не рекомендую", собранные данные отлично подходят для бинарной классификации и уже размечены

##2. Предобработка

In [ ]:
import pandas as pd
df = pd.read_csv("baldurs_gate_3_reviews_balanced.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   recommend    1000 non-null   bool  
 1   review_text  1000 non-null   object
dtypes: bool(1), object(1)
memory usage: 8.9+ KB


In [ ]:
df['recommend'] = df['recommend'].astype(int)
df['review_text'] = df['review_text'].astype(str)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   recommend    1000 non-null   int64 
 1   review_text  1000 non-null   object
dtypes: int64(1), object(1)
memory usage: 15.8+ KB


In [ ]:
df.head()

,recommend,review_text
0,1,"perfect story, perfect character customizaton,..."
1,0,Very sweaty D&D game. If you have any sort of...
2,1,"""No one back home is going to ever believe thi..."
3,0,"A great game overall in terms of production, b..."
4,0,"Not my style of game, I'm sure it has its nich..."


In [ ]:
df['review_text'].apply(lambda x: x.lower())

,review_text
0,"perfect story, perfect character customizaton,..."
1,very sweaty d&d game. if you have any sort of...
2,"""no one back home is going to ever believe thi..."
3,"a great game overall in terms of production, b..."
4,"not my style of game, i'm sure it has its nich..."
...,...
995,"played dos and dos2, no issues. bg3 is the onl..."
996,baldur's gate 3 ruined rpgs for me.\r\n\r\naft...
997,it's like divinity: original sin 2 if divinity...
998,"larian ""patched"" the game and now it wont star..."


## 3. Векторизация и метрики

Векторизуем и поверяем, что всё хорошо векторизовалось

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['review_text']).toarray()

In [ ]:
X[0]

array([0., 0., 0., ..., 0., 0., 0.])

In [ ]:
X.shape # 1000 текстов и 5000 фичей TF-IDF (уникальных слов в корпусе)

(1000, 5000)

In [ ]:
X[0][1500:1600]

array([0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.     

In [ ]:
y = df['recommend'].values
print(y)

[1 0 1 0 0 1 1 0 0 1 1 0 1 0 0 1 0 0 0 1 1 1 1 0 0 1 0 0 1 0 0 1 0 0 1 0 1
 1 1 1 1 0 0 1 0 0 0 1 1 1 0 0 0 0 1 0 0 1 1 0 1 0 0 1 1 1 0 0 0 1 1 0 0 1
 0 1 0 0 0 0 0 1 0 0 1 0 1 1 1 0 0 0 1 1 1 1 0 0 1 1 0 0 1 1 1 1 1 0 1 1 1
 1 1 1 1 1 0 1 0 0 1 0 0 1 1 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 1 0 1 1 1 0
 0 1 1 1 0 1 1 0 0 1 0 0 0 1 1 1 1 1 1 0 0 0 0 0 1 0 0 0 1 0 0 1 1 1 0 0 0
 1 0 1 0 1 0 0 0 0 1 0 0 0 1 1 0 0 0 1 0 1 1 0 1 0 0 0 1 0 1 1 1 0 0 0 0 0
 1 0 1 1 1 1 0 0 1 0 1 1 0 1 0 0 1 0 1 1 1 1 1 0 0 0 1 1 1 1 1 1 0 1 1 1 0
 1 0 0 0 0 0 0 1 1 0 1 0 1 1 0 0 1 0 0 1 1 1 1 0 0 1 0 1 1 1 1 1 0 0 1 0 0
 1 0 1 0 0 0 0 1 0 0 0 0 0 1 1 0 0 1 1 1 1 1 1 0 0 1 1 0 1 0 0 0 0 0 0 0 1
 0 1 0 0 0 0 0 0 1 1 0 1 1 1 1 1 0 0 1 1 0 1 0 0 1 0 1 1 0 1 0 1 1 1 0 1 1
 0 0 1 1 0 1 0 0 0 1 0 1 1 1 0 1 1 1 1 0 1 0 0 1 0 0 1 0 0 1 0 0 0 1 1 0 0
 1 1 1 0 1 0 1 1 1 0 1 1 0 1 1 0 1 1 0 0 0 0 0 0 0 0 1 0 1 1 0 0 1 0 0 0 1
 0 1 1 1 0 1 0 1 1 0 1 1 1 1 1 1 0 0 0 0 1 1 0 0 0 0 1 1 0 1 0 0 0 0 1 1 1
 0 0 1 1 1 0 1 0 1 0 0 0 

## 4. Оборачиваем в датасет

In [ ]:
import torch

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

In [ ]:
X_tensor[0], y_tensor[0]

(tensor([0., 0., 0.,  ..., 0., 0., 0.]), tensor(1))

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

dataset_torch = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset_torch, batch_size=32, shuffle=True)

In [ ]:
dataset_torch # контейнер данных

In [ ]:
dataloader # итератор: он оборачивает Dataset и добавляет batching, shuffling, parallel loading

Создайте дата-лоадер для своих данных

## 5. Обучение

In [ ]:
import torch.nn as nn
import torch.optim as optim
import numpy as np

class CNN1D(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * (input_dim // 4), 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.unsqueeze(1) # добавляет указание на кол-во каналов channels, в RGB их 3, тут 1 (batch_size, channels, sequence_length)
        # было: (batch_size, input_dim) напр., (32, 1000)
        # стало: (batch_size, 1, input_dim) напр., (32, 1, 1000)
        x = self.pool(self.relu(self.conv1(x))) # здесь видно, что пулинг вызывает дважды!
        x = self.pool(self.relu(self.conv2(x))) # здесь видно, что пулинг вызывает дважды!
        x = x.view(x.size(0), -1) # это операция flatten
        x = self.relu(self.fc1(x))
        return self.fc2(x)

model = CNN1D(X.shape[1], len(np.unique(y)))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    total_loss = 0
    for batch_x, batch_y in dataloader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")

Epoch 1, Loss: 0.9385
Epoch 2, Loss: 0.6966
Epoch 3, Loss: 0.6947
Epoch 4, Loss: 0.6934
Epoch 5, Loss: 0.6939


Тут моделька переобучается, и результаты не очень. Поэтому я подкрутила параметры, чтобы улучшить качество. Эти параметры были подобраны путём тестов и расчётов дипсика.

## 6. Получаем предсказания

In [ ]:
import torch.nn as nn
import torch.optim as optim
import numpy as np
import torch

class CNN1D(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=4, padding=1)
        self.relu = nn.ReLU()

        # Dynamically calculate the input size for the first fully connected layer
        # This ensures robustness against changes in kernel_size, padding, or pooling
        dummy_input = torch.zeros(1, 1, input_dim) # (batch_size, channels, sequence_length)
        x = self.pool(self.relu(self.conv1(dummy_input)))
        x = self.pool(self.relu(self.conv2(x)))
        fc_input_features = x.view(x.size(0), -1).size(1)

        self.fc1 = nn.Linear(fc_input_features, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1) # добавляет указание на кол-во каналов channels, в RGB их 3, тут 1 (batch_size, channels, sequence_length)
        # было: (batch_size, input_dim) напр., (32, 1000)
        # стало: (batch_size, 1, input_dim) напр., (32, 1, 1000)
        x = self.pool(self.relu(self.conv1(x))) # здесь видно, что пулинг вызывает дважды!
        x = self.pool(self.relu(self.conv2(x))) # здесь видно, что пулинг вызывает дважды!
        x = x.view(x.size(0), -1) # это операция flatten
        x = self.relu(self.fc1(x))
        return self.fc2(x)

model = CNN1D(X.shape[1], len(np.unique(y)))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(7):
    total_loss = 0
    for batch_x, batch_y in dataloader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")

Epoch 1, Loss: 1.1115
Epoch 2, Loss: 0.6952
Epoch 3, Loss: 0.6912
Epoch 4, Loss: 0.6704
Epoch 5, Loss: 0.4689
Epoch 6, Loss: 0.2455
Epoch 7, Loss: 0.1356


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
torch.save(model.state_dict(), '/content/drive/MyDrive/model.pt')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# @title
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
with torch.no_grad():
    predictions = model(X_tensor)
    _, predicted = torch.max(predictions, 1)

In [ ]:
print(classification_report(y, predicted.numpy()))
print(confusion_matrix(y, predicted.numpy()))

              precision    recall  f1-score   support

           0       0.99      0.98      0.99       500
           1       0.98      0.99      0.99       500

    accuracy                           0.99      1000
   macro avg       0.99      0.99      0.99      1000
weighted avg       0.99      0.99      0.99      1000

[[490  10]
 [  4 496]]


P.S. в этот раз блокнот получился с большим числом черновых вариантов кода и пояснений, потому что я потратила много времени на эту работу и хотела показать, что и как у меня получилось :)